# GlassDesign ML 1.0 — самостоятельный запуск

**Подбор составов оксидных стёкол по заданным свойствам.** Публичная версия использует расчётное ядро 391 и интерфейс 409.

1. Выберите **Среда выполнения → Выполнить все**.
2. При первом запуске Colab скачает код из GitHub, 4 обученные модели из открытых авторских файлов Google Drive и необходимые библиотеки.
3. По возможности будет построен справочный индекс SciGlass из открытой базы GlassPy. Это может потребовать дополнительных ресурсов; если загрузка недоступна, расчётный генератор продолжит работать, но литературный раздел будет ограничен.
4. В последней ячейке откройте ссылку `trycloudflare.com`. Сеанс Colab должен оставаться активным.

**Ограничение:** рейтинг A/B/C/D из полной исследовательской версии зависит от нераспространяемых обучающих/справочных таблиц и здесь не вычисляется. Согласие двух моделей и опубликованные аналоги не являются экспериментальным подтверждением результата. Адрес сайта временный; скачанные файлы остаются в сеансе Colab. Запускайте код и файлы моделей только из доверенного репозитория.

Исходники: https://github.com/polinashidlovskaya/glassdesign-ml

In [ ]:
# Модели распространяются автором через файлы Google Drive с доступом по ссылке.
MODEL_FILE_IDS = {
    'density_extra_trees_final.joblib': '1a8FO6SmxLygKN-c2OrWJDKR-OjdxQXdu',
    'refractive_index_extra_trees_final_v2.joblib': '17y7tKUddZHSYbecl344C3XFm6CTs9Vt-',
    'rf_tg_final.joblib': '1Cn4dSABuMloTtLa15fGKUn21iONmIc7f',
    'extra_trees_softening_final.joblib': '19CDRPCMfCYGUBaGhjLa6iJMBPlPPG_GM',
}
# В окончательной ссылке из курсовой используется main.
# Для проверки ещё не объединённого изменения временно укажите ветку release/.
RELEASE_REF = 'main'
assert all(MODEL_FILE_IDS.values())

In [ ]:
# 1. Код, зависимости, модели; доступ к личному Google Drive НЕ нужен.
import os, sys, subprocess, pathlib
project = pathlib.Path('/content/glassdesign-ml')
repo_url = 'https://github.com/polinashidlovskaya/glassdesign-ml.git'
if not project.exists():
    subprocess.run(['git','clone','--depth','1','--branch',RELEASE_REF,repo_url,str(project)],check=True)
else:
    if not (project/'.git').is_dir():
        raise RuntimeError('Папка проекта занята посторонними файлами. Перезапустите среду Colab.')
    if subprocess.run(['git','-C',str(project),'status','--porcelain','--untracked-files=no'],capture_output=True,text=True,check=True).stdout.strip():
        raise RuntimeError('Есть локальные изменения исходников. Используйте новую среду Colab.')
    subprocess.run(['git','-C',str(project),'fetch','--depth','1','origin',RELEASE_REF],check=True)
    subprocess.run(['git','-C',str(project),'reset','--hard','FETCH_HEAD'],check=True)

subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(project/'requirements.txt'),'gdown','scipy'],check=True)
import gdown
ROOT = project/'runtime'
folder = ROOT/'GlassDesignML'/'models'
folder.mkdir(parents=True,exist_ok=True)
for filename, file_id in MODEL_FILE_IDS.items():
    target=folder/filename
    if not target.is_file() or target.stat().st_size<1024*1024:
        gdown.download(id=file_id,output=str(target),quiet=False)
    if not target.is_file() or target.stat().st_size<1024*1024:
        raise RuntimeError('Не удалось загрузить модель: '+filename+'; проверьте доступ по ссылке.')
    print('✅',filename,round(target.stat().st_size/1024**2,2),'МиБ')
os.environ['GLASSDESIGN_ROOT']=str(ROOT)
os.environ['GLASSDESIGN_PUBLIC_DEMO']='1'
sys.path.insert(0,str(project))
print('✅ Подготовлены модели и исходники GlassDesign ML 1.0')

In [ ]:
# 2. Подготовка открытых экспериментальных данных SciGlass (необязательно для генерации).
# Данные SciGlass — справочник публикаций, а НЕ независимая проверка ML.
import pandas as pd, gc, traceback
from glass398_universal_literature import (
    PROPERTIES, make_measurement_arrays, save_measurement_index
)
index_file = ROOT/'GlassDesignML_Upgrade'/'SciGlass_Universal_Literature_step398'/'cache'/'sciglass_four_properties_mol_398_1.npz'
if index_file.is_file():
    print('✅ Индекс SciGlass уже построен')
else:
    try:
        from glasspy.data import SciGlass
        available=SciGlass.available_properties()
        props=list(PROPERTIES)
        missing=set(props)-set(available)
        if missing:
            raise RuntimeError('В версии GlassPy отсутствуют свойства: '+str(sorted(missing)))
        sc=SciGlass(properties_cfg={'keep':props,'drop':[p for p in available if p not in props]})
        raw=sc.data
        comp=raw['compounds']
        measured=raw['property']
        metadata=raw['metadata'] if 'metadata' in raw else pd.DataFrame(index=raw.index)
        # Техническая строка лишь задаёт форму массива для универсального индекса.
        reference=pd.DataFrame([{'Candidate_ID':1,'SiO2':70.,'Na2O':15.,'CaO':15.}])
        arrays=make_measurement_arrays(comp,measured,metadata,reference)
        save_measurement_index(arrays,index_file)
        print('✅ Индекс SciGlass создан:',len(arrays['X']),'записей')
        del sc,raw,comp,measured,metadata,arrays
        gc.collect()
    except Exception as exc:
        print('⚠️ Индекс SciGlass не удалось построить. Генератор доступен, литературная часть может отсутствовать.')
        print(type(exc).__name__+': '+str(exc)[:700])
print('Продолжаем запуск GlassDesign ML 1.0')

In [ ]:
# 3. Запуск сайта с временной публичной ссылкой Cloudflare.
import time, re, requests, urllib.request, socket
from IPython.display import display, HTML
port='8501'
log_path=pathlib.Path('/content/glassdesign_streamlit.log')
env=os.environ.copy()
env['GLASSDESIGN_ROOT']=str(ROOT)
env['GLASSDESIGN_PUBLIC_DEMO']='1'

# При повторном исполнении ячейки корректно перезапустим только текущий процесс.
if 'app_process' in globals() and app_process.poll() is None:
    app_process.terminate()
    app_process.wait(timeout=15)
with log_path.open('w') as log:
    app_process=subprocess.Popen(
        [sys.executable,'-m','streamlit','run',str(project/'app_glassdesign_409.py'),
         '--server.port='+port,'--server.address=0.0.0.0',
         '--server.headless=true','--server.fileWatcherType=none',
         '--browser.gatherUsageStats=false'],
        cwd=str(project),env=env,stdout=log,stderr=subprocess.STDOUT)
for _ in range(90):
    if app_process.poll() is not None:
        raise RuntimeError('Ошибка Streamlit:\\n'+log_path.read_text(errors='replace')[-3000:])
    try:
        if requests.get('http://127.0.0.1:'+port+'/_stcore/health',timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError('Streamlit не запустился:\\n'+log_path.read_text(errors='replace')[-3000:])

cf=pathlib.Path('/content/cloudflared')
if not cf.is_file():
    url='https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    with requests.get(url,stream=True,timeout=120) as response:
        response.raise_for_status()
        with cf.open('wb') as output:
            for part in response.iter_content(1024*1024):
                if part: output.write(part)
    cf.chmod(0o755)
if 'tunnel_process' in globals() and tunnel_process.poll() is None:
    tunnel_process.terminate()
    tunnel_process.wait(timeout=10)
tunnel_log=pathlib.Path('/content/glassdesign_cloudflare.log')
with tunnel_log.open('w') as log:
    tunnel_process=subprocess.Popen(
        [str(cf),'tunnel','--no-autoupdate','--protocol','http2',
         '--url','http://127.0.0.1:'+port],stdout=log,stderr=subprocess.STDOUT)
public_url=None
for _ in range(90):
    output=tunnel_log.read_text(errors='replace')
    match=re.search(r'https://[a-zA-Z0-9-]+\\.trycloudflare\\.com',output)
    if match:
        public_url=match.group(0)
        break
    if tunnel_process.poll() is not None:
        break
    time.sleep(2)
if not public_url:
    raise RuntimeError('Не удалось получить Cloudflare URL:\\n'+tunnel_log.read_text(errors='replace')[-2500:])
print('\\n✅ GLASSDESIGN ML ОТКРЫТ')
print('Временная ссылка:',public_url)
print('Оставьте сеанс Google Colab подключённым, пока используете приложение.')
display(HTML('<a href="'+public_url+'" target="_blank" style="font-size:22px;font-weight:bold">🌐 Открыть GlassDesign ML 1.0</a>'))